# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click !here goes the icon of the corresponding button in the gutter! button.
To debug a cell, press Alt+Shift+Enter, or click !here goes the icon of the corresponding button in the gutter! button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/jupyter-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [1]:
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras import models, layers
from sklearn.metrics import classification_report

import os
import numpy as np
import csv
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras import models, layers
from sklearn.metrics import classification_report
from tensorflow.python.keras.callbacks import EarlyStopping

# === CONFIG ===
NpyFolder = "Hand_LandMark"
LabelsFile = "labels.csv"
NUM_AUGMENTATIONS = 5

# === LOAD DATA + AUGMENT ===
X = []
y = []

def augment_landmarks(landmarks):
    # Small Gaussian noise
    noise = np.random.normal(0, 0.01, size=landmarks.shape)
    augmented = landmarks + noise
    return augmented

with open(LabelsFile, mode="r") as f:
    reader = csv.reader(f)
    next(reader)  # skip header
    for row in reader:
        npy_filename, label = row
        npy_path = os.path.join(NpyFolder, npy_filename)

        if not os.path.exists(npy_path):
            continue

        landmarks = np.load(npy_path)  # shape: (frames,126)

        # Process original
        mean_features = np.mean(landmarks, axis=0)
        std_features = np.std(landmarks, axis=0)
        min_features = np.min(landmarks, axis=0)
        max_features = np.max(landmarks, axis=0)
        all_features = np.concatenate([mean_features, std_features, min_features, max_features])

        X.append(all_features)
        y.append(label)

        # Generate augmented samples
        for i in range(NUM_AUGMENTATIONS):
            aug_landmarks = augment_landmarks(landmarks)
            mean_features = np.mean(aug_landmarks, axis=0)
            std_features = np.std(aug_landmarks, axis=0)
            min_features = np.min(aug_landmarks, axis=0)
            max_features = np.max(aug_landmarks, axis=0)
            all_features = np.concatenate([mean_features, std_features, min_features, max_features])

            X.append(all_features)
            y.append(label)

print("✅ Loaded original + augmented samples:", len(X))
print("Labels distribution:", Counter(y))


# Check your dataset size
print("✅ Final dataset size:", len(X))
print("✅ Label distribution:", Counter(y))

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes = len(le.classes_)
print("✅ Encoded labels into", num_classes, "classes.")

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=42
)
print("✅ Train size:", len(X_train), "Test size:", len(X_test))

# Build the model
model = models.Sequential([
    layers.Input(shape=(X_train.shape[1],)),  # 504 features
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation="softmax")
])

# Compile
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Early stopping callback
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

# Train
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

# Evaluate
loss, acc = model.evaluate(X_test, y_test)
print("🎯 Test Accuracy:", acc)

# Predict
pred_probs = model.predict(X_test)
pred_classes = pred_probs.argmax(axis=1)

# Report
print("🧠 Classification Report:")
print(classification_report(y_test, pred_classes, target_names=le.classes_))



✅ Loaded original + augmented samples: 3324
Labels distribution: Counter({'can i help you': 48, 'go and sleep': 48, 'you are bad': 48, 'are you hiding something': 42, 'bring water for me': 42, 'comb your hair': 42, 'congratulations': 42, 'do me a favour': 42, 'do not hurt me': 42, 'do not worry': 42, 'had your food': 42, 'help me': 42, 'how old are you': 42, 'i am afraid of that': 42, 'i am crying': 42, 'i am feeling cold': 42, 'i am hungry': 42, 'i am so sorry to hear that': 42, 'i enjoyed a lot': 42, 'i got hurt': 42, 'i need water': 42, 'i really appreciate it': 42, 'i somehow got to know about it': 42, 'my name is xxxxxxxx': 42, 'speak softly': 42, 'turn on light turn off light': 42, 'what did you tell him': 42, 'what do you think': 42, 'what you want': 42, 'who are you': 42, 'why are you angry': 42, 'why are you crying': 42, 'why are you disappointed': 42, 'you can do it': 42, 'can you repeat that please': 36, 'do not make me angry': 36, 'do not take it to the heart': 36, 'he is o

In [8]:
import cv2
import mediapipe as mp
import numpy as np

# === CONFIG ===
VIDEO_PATH = "ML/can you repeat that please (4).MP4"  # Replace with your video path

# === Load your trained model, scaler, and label encoder ===
# Make sure you have them loaded in this script:
# model = tf.keras.models.load_model("model_path")
# scaler = joblib.load("scaler.pkl")
# le = joblib.load("label_encoder.pkl")

# Mediapipe setup
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,  # TWO HANDS
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

cap = cv2.VideoCapture(VIDEO_PATH)

all_landmarks = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(frame_rgb)

    if results.multi_hand_landmarks:
        # Start empty
        frame_landmarks = []

        # For each detected hand (up to 2)
        for hand_landmarks in results.multi_hand_landmarks:
            for lm in hand_landmarks.landmark:
                frame_landmarks.extend([lm.x, lm.y, lm.z])

        # If only ONE hand detected, pad the rest with zeros
        if len(results.multi_hand_landmarks) == 1:
            frame_landmarks.extend([0.0] * 63)  # Pad missing 21 landmarks

        all_landmarks.append(frame_landmarks)

cap.release()

# Handle case no landmarks detected
if len(all_landmarks) == 0:
    print("❌ No hand landmarks detected in this video.")
else:
    landmarks_array = np.array(all_landmarks)

    # Compute statistical features
    mean_features = np.mean(landmarks_array, axis=0)
    std_features = np.std(landmarks_array, axis=0)
    min_features = np.min(landmarks_array, axis=0)
    max_features = np.max(landmarks_array, axis=0)

    all_features = np.concatenate([mean_features, std_features, min_features, max_features])  # Shape: (504,)

    # Scale the features
    all_features_scaled = scaler.transform([all_features])

    # Predict
    pred_probs = model.predict(all_features_scaled)
    pred_class = pred_probs.argmax(axis=1)[0]
    pred_label = le.inverse_transform([pred_class])[0]

    print(f"🎯 Predicted Label: {pred_label}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 333ms/step
🎯 Predicted Label: i got hurt


In [2]:
from deepface import DeepFace
import cv2
def analyze_emotions_from_video(video_path, frame_sample_rate=10):
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    emotions = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % frame_sample_rate == 0:
            try:
                result = DeepFace.analyze(frame, actions=['emotion'], enforce_detection=False)
                dominant_emotion = result[0]['dominant_emotion']
                print(f"Frame {frame_count}: {dominant_emotion}")
                emotions.append(dominant_emotion)
            except Exception as e:
                print(f"Error at frame {frame_count}: {e}")

        frame_count += 1

    cap.release()

    # Summarize emotions
    if not emotions:
        return None  # or return "neutral", etc.

    emotion_summary = Counter(emotions)
    most_common_emotion = emotion_summary.most_common(1)[0][0]

    return most_common_emotion
analyze_emotions_from_video("D:/UTILITAIRE POUR TRAVAIL/PROJECT/PythonProject/NLP_TEST/DEAF-P AI_BIG_DATA/data/iSign-videos_v1.1/_1tafiQHMEs--2.mp4")

Frame 0: angry
Frame 10: angry
Frame 20: neutral
Frame 30: angry
Frame 40: angry
Frame 50: neutral
Frame 60: neutral


NameError: name 'Counter' is not defined

In [13]:
import customtkinter as ctk
import tkinter as tk
from math import cos, sin, pi

def draw_meter(canvas, score=4):
    canvas.delete("all")
    cx, cy = 200, 200
    r = 140
    width = 30

    # Draw color segments for the arc manually
    segments = 100  # More segments = smoother gradient
    for i in range(segments):
        angle_deg = 180 + (i * 180 / segments)
        angle_rad = angle_deg * pi / 180
        x0 = cx + r * cos(angle_rad)
        y0 = cy + r * sin(angle_rad)
        x1 = cx + (r - width) * cos(angle_rad)
        y1 = cy + (r - width) * sin(angle_rad)

        # Gradient color between green (0), yellow (middle), red (end)
        red = int(min(255, (i / segments) * 510))
        green = int(min(255, 510 - (i / segments) * 510))
        color = f"#{red:02x}{green:02x}00"
        canvas.create_line(x0, y0, x1, y1, fill=color, width=3)

    # Draw pointer based on score (0 to 4)
    angle_step = pi / 4  # 5 moods, so 4 steps
    angle = pi + score * angle_step
    pointer_length = r - width - 10
    px = cx + pointer_length * cos(angle)
    py = cy + pointer_length * sin(angle)

    canvas.create_line(cx, cy, px, py, fill="black", width=8, capstyle="round")
    canvas.create_oval(cx - 8, cy - 8, cx + 8, cy + 8, fill="black", outline="gray", width=2)

def on_score_change(choice):
    draw_meter(canvas, int(choice))

# === CustomTkinter App
ctk.set_appearance_mode("Light")
ctk.set_default_color_theme("blue")
root = ctk.CTk()
root.geometry("400x400")
root.title("Color Gradient Mood Meter")

canvas = tk.Canvas(root, width=400, height=300, bg="white", highlightthickness=0)
canvas.pack(pady=20)

# Dropdown to choose mood level
score_var = ctk.StringVar(value="4")
dropdown = ctk.CTkOptionMenu(root, variable=score_var, values=[str(i) for i in range(5)],
                             command=on_score_change)
dropdown.pack()

draw_meter(canvas, int(score_var.get()))
root.mainloop()


In [15]:
import customtkinter as ctk
import tkinter as tk
from math import cos, sin, pi

def draw_meter(canvas, score=4):
    canvas.delete("all")
    cx, cy = 200, 200
    r = 140
    width = 30

    # Smooth color gradient across 100 segments
    segments = 500
    for i in range(segments):
        angle_deg = 180 + (i * 180 / segments)
        angle_rad = angle_deg * pi / 180
        x0 = cx + r * cos(angle_rad)
        y0 = cy + r * sin(angle_rad)
        x1 = cx + (r - width) * cos(angle_rad)
        y1 = cy + (r - width) * sin(angle_rad)

        # Smooth RGB gradient from green → yellow → red
        t = i / (segments - 1)
        if t < 0.5:
            # Green to Yellow
            r_val = int(255 * (2 * t))
            g_val = 255
        else:
            # Yellow to Red
            r_val = 255
            g_val = int(255 * (2 - 2 * t))
        color = f"#{r_val:02x}{g_val:02x}00"

        canvas.create_line(x0, y0, x1, y1, fill=color, width=3)

    # Pointer (0–4)
    angle_step = pi / 4
    angle = pi + score * angle_step
    pointer_length = r - width - 10
    px = cx + pointer_length * cos(angle)
    py = cy + pointer_length * sin(angle)

    # Draw pointer
    canvas.create_line(cx, cy, px, py, fill="white", width=6, capstyle="round")
    canvas.create_oval(cx - 7, cy - 7, cx + 7, cy + 7, fill="white", outline="gray", width=2)

def on_score_change(choice):
    draw_meter(canvas, int(choice))

# === App Setup
ctk.set_appearance_mode("Dark")
ctk.set_default_color_theme("dark-blue")

root = ctk.CTk()
root.title("Mood Meter")
root.geometry("420x420")
root.configure(fg_color="black")

canvas = tk.Canvas(root, width=400, height=300, bg="black", highlightthickness=0)
canvas.pack(pady=20)

# Score selection dropdown
score_var = ctk.StringVar(value="2")
dropdown = ctk.CTkOptionMenu(root, variable=score_var, values=[str(i) for i in range(5)],
                             command=on_score_change,
                             fg_color="#222", text_color="white", button_color="#444")
dropdown.pack()

# Initial draw
draw_meter(canvas, int(score_var.get()))
root.mainloop()


In [ ]:
import customtkinter as ctk
import tkinter as tk
from tkinter import filedialog, messagebox
import cv2
import numpy as np
from PIL import Image, ImageTk
import threading
import time
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.figure import Figure
import seaborn as sns

# Set appearance mode and color theme
ctk.set_appearance_mode("dark")
ctk.set_default_color_theme("blue")

class AIAnalysisDashboard:
    def __init__(self):
        self.root = ctk.CTk()
        self.root.title("DEAF-P AI - Healthcare Assistant")
        self.root.geometry("1400x900")
        self.root.configure(fg_color="#0f172a")  # Dark background

        # Analysis data
        self.analysis_data = {
            "emotion": "neutral",
            "confidence": 0,
            "translation": "",
            "depression_score": 2,
            "is_processing": False
        }

        # Video capture
        self.cap = None
        self.video_thread = None
        self.is_recording = False

        self.setup_ui()

    def setup_ui(self):
        # Main container with scrollable frame
        self.main_frame = ctk.CTkScrollableFrame(
            self.root,
            fg_color="#0f172a",
            scrollbar_button_color="#334155",
            scrollbar_button_hover_color="#475569"
        )
        self.main_frame.pack(fill="both", expand=True, padx=20, pady=20)

        self.create_header()
        self.create_hero_section()
        self.create_main_interface()
        self.create_features_section()

    def create_header(self):
        # Header frame
        header_frame = ctk.CTkFrame(
            self.main_frame,
            fg_color="#1e293b",
            corner_radius=12,
            height=80
        )
        header_frame.pack(fill="x", pady=(0, 20))
        header_frame.pack_propagate(False)

        # Logo and title
        title_label = ctk.CTkLabel(
            header_frame,
            text="🧠 DEAF-P AI",
            font=ctk.CTkFont(family="Montserrat", size=24, weight="bold"),
            text_color="#0ea5e9"
        )
        title_label.pack(side="left", padx=20, pady=20)

        # Navigation buttons
        nav_frame = ctk.CTkFrame(header_frame, fg_color="transparent")
        nav_frame.pack(side="right", padx=20, pady=20)

        nav_buttons = ["Dashboard", "Analysis", "Settings", "Help"]
        for btn_text in nav_buttons:
            btn = ctk.CTkButton(
                nav_frame,
                text=btn_text,
                fg_color="transparent",
                text_color="#e2e8f0",
                hover_color="#334155",
                width=80,
                height=32
            )
            btn.pack(side="left", padx=5)

    def create_hero_section(self):
        # Hero section
        hero_frame = ctk.CTkFrame(
            self.main_frame,
            fg_color="#1e293b",
            corner_radius=12
        )
        hero_frame.pack(fill="x", pady=(0, 20))

        # Title and description
        title = ctk.CTkLabel(
            hero_frame,
            text="AI for Non-Verbal Communication",
            font=ctk.CTkFont(family="Montserrat", size=36, weight="bold"),
            text_color="#e2e8f0"
        )
        title.pack(pady=(30, 10))

        description = ctk.CTkLabel(
            hero_frame,
            text="Advanced facial and gestural emotion analysis system designed to support\nthe welfare and communication needs of deaf-mute individuals in society.",
            font=ctk.CTkFont(size=16),
            text_color="#94a3b8"
        )
        description.pack(pady=(0, 30))

        # Stats cards
        stats_frame = ctk.CTkFrame(hero_frame, fg_color="transparent")
        stats_frame.pack(pady=(0, 30))

        stats_data = [
            ("🧠", "98.5%", "Emotion Accuracy", "#a855f7"),
            ("⚡", "Real-time", "Analysis Speed", "#ec4899"),
            ("👥", "10K+", "Users Supported", "#0ea5e9")
        ]

        for i, (icon, value, label, color) in enumerate(stats_data):
            card = ctk.CTkFrame(
                stats_frame,
                fg_color="#334155",
                corner_radius=8,
                width=200,
                height=120
            )
            card.grid(row=0, column=i, padx=20, pady=10)
            card.pack_propagate(False)

            icon_label = ctk.CTkLabel(
                card,
                text=icon,
                font=ctk.CTkFont(size=24),
                text_color=color
            )
            icon_label.pack(pady=(15, 5))

            value_label = ctk.CTkLabel(
                card,
                text=value,
                font=ctk.CTkFont(family="Montserrat", size=20, weight="bold"),
                text_color="#e2e8f0"
            )
            value_label.pack()

            label_label = ctk.CTkLabel(
                card,
                text=label,
                font=ctk.CTkFont(size=12),
                text_color="#94a3b8"
            )
            label_label.pack(pady=(0, 15))

    def create_main_interface(self):
        # Tabbed interface
        self.tabview = ctk.CTkTabview(
            self.main_frame,
            fg_color="#1e293b",
            segmented_button_fg_color="#334155",
            segmented_button_selected_color="#0ea5e9",
            segmented_button_selected_hover_color="#0284c7",
            text_color="#e2e8f0"
        )
        self.tabview.pack(fill="both", expand=True, pady=(0, 20))

        # Add tabs
        self.tabview.add("📹 Video Analysis")
        self.tabview.add("🎤 Live Translation")

        self.setup_video_analysis_tab()
        self.setup_live_translation_tab()

    def setup_video_analysis_tab(self):
        tab = self.tabview.tab("📹 Video Analysis")

        # Main content frame
        content_frame = ctk.CTkFrame(tab, fg_color="transparent")
        content_frame.pack(fill="both", expand=True, padx=20, pady=20)

        # Left side - Video upload
        left_frame = ctk.CTkFrame(
            content_frame,
            fg_color="#334155",
            corner_radius=12
        )
        left_frame.pack(side="left", fill="both", expand=True, padx=(0, 10))

        # Video upload section
        upload_title = ctk.CTkLabel(
            left_frame,
            text="Video Upload & Analysis",
            font=ctk.CTkFont(family="Montserrat", size=18, weight="bold"),
            text_color="#e2e8f0"
        )
        upload_title.pack(pady=(20, 10))

        # Drag and drop area
        self.upload_frame = ctk.CTkFrame(
            left_frame,
            fg_color="#1e293b",
            corner_radius=8,
            height=300,
            border_width=2,
            border_color="#475569"
        )
        self.upload_frame.pack(fill="x", padx=20, pady=10)
        self.upload_frame.pack_propagate(False)

        upload_icon = ctk.CTkLabel(
            self.upload_frame,
            text="📁",
            font=ctk.CTkFont(size=48),
            text_color="#94a3b8"
        )
        upload_icon.pack(pady=(60, 10))

        upload_text = ctk.CTkLabel(
            self.upload_frame,
            text="Drag and drop your video file here\nor click to browse",
            font=ctk.CTkFont(size=14),
            text_color="#94a3b8"
        )
        upload_text.pack()

        # Upload button
        upload_btn = ctk.CTkButton(
            self.upload_frame,
            text="Choose File",
            command=self.upload_video,
            fg_color="#0ea5e9",
            hover_color="#0284c7",
            font=ctk.CTkFont(weight="bold")
        )
        upload_btn.pack(pady=20)

        # Progress bar
        self.progress_bar = ctk.CTkProgressBar(
            left_frame,
            progress_color="#0ea5e9",
            fg_color="#1e293b"
        )
        self.progress_bar.pack(fill="x", padx=20, pady=10)
        self.progress_bar.set(0)

        # Analyze button
        self.analyze_btn = ctk.CTkButton(
            left_frame,
            text="🔍 Analyze Video",
            command=self.analyze_video,
            fg_color="#a855f7",
            hover_color="#9333ea",
            font=ctk.CTkFont(weight="bold"),
            height=40,
            state="disabled"
        )
        self.analyze_btn.pack(fill="x", padx=20, pady=(10, 20))

        # Right side - Results
        right_frame = ctk.CTkFrame(
            content_frame,
            fg_color="transparent"
        )
        right_frame.pack(side="right", fill="y", padx=(10, 0))

        self.create_results_panel(right_frame)

    def create_results_panel(self, parent):
        # Translation output
        translation_frame = ctk.CTkFrame(
            parent,
            fg_color="#334155",
            corner_radius=12,
            width=350
        )
        translation_frame.pack(fill="x", pady=(0, 15))
        translation_frame.pack_propagate(False)

        translation_title = ctk.CTkLabel(
            translation_frame,
            text="💬 Translation Output",
            font=ctk.CTkFont(family="Montserrat", size=16, weight="bold"),
            text_color="#e2e8f0"
        )
        translation_title.pack(pady=(15, 10))

        self.translation_text = ctk.CTkTextbox(
            translation_frame,
            height=100,
            fg_color="#1e293b",
            text_color="#e2e8f0",
            font=ctk.CTkFont(size=12)
        )
        self.translation_text.pack(fill="x", padx=15, pady=(0, 15))
        self.translation_text.insert("0.0", "Translation will appear here...")

        # Emotion display
        emotion_frame = ctk.CTkFrame(
            parent,
            fg_color="#334155",
            corner_radius=12,
            width=350
        )
        emotion_frame.pack(fill="x", pady=(0, 15))
        emotion_frame.pack_propagate(False)

        emotion_title = ctk.CTkLabel(
            emotion_frame,
            text="😊 Emotion Analysis",
            font=ctk.CTkFont(family="Montserrat", size=16, weight="bold"),
            text_color="#e2e8f0"
        )
        emotion_title.pack(pady=(15, 10))

        self.emotion_label = ctk.CTkLabel(
            emotion_frame,
            text="Neutral",
            font=ctk.CTkFont(size=24, weight="bold"),
            text_color="#0ea5e9"
        )
        self.emotion_label.pack()

        self.confidence_label = ctk.CTkLabel(
            emotion_frame,
            text="Confidence: 0%",
            font=ctk.CTkFont(size=12),
            text_color="#94a3b8"
        )
        self.confidence_label.pack(pady=(0, 15))

        # Depression meter
        self.create_depression_meter(parent)

    def create_depression_meter(self, parent):
        meter_frame = ctk.CTkFrame(
            parent,
            fg_color="#334155",
            corner_radius=12,
            width=350,
            height=250
        )
        meter_frame.pack(fill="x")
        meter_frame.pack_propagate(False)

        meter_title = ctk.CTkLabel(
            meter_frame,
            text="📊 Depression Analysis",
            font=ctk.CTkFont(family="Montserrat", size=16, weight="bold"),
            text_color="#e2e8f0"
        )
        meter_title.pack(pady=(15, 10))

        # Create matplotlib figure for depression meter
        fig = Figure(figsize=(4, 2.5), facecolor='#334155')
        ax = fig.add_subplot(111, facecolor='#334155')

        # Create gauge chart
        categories = ['Minimal', 'Mild', 'Moderate', 'Severe', 'Critical']
        values = [1, 1, 1, 1, 1]
        colors = ['#10b981', '#f59e0b', '#f97316', '#ef4444', '#dc2626']

        # Current score (example: 2 = Mild)
        current_score = 2
        highlight_colors = ['#374151' if i != current_score else colors[i] for i in range(5)]

        bars = ax.barh(categories, values, color=highlight_colors)
        ax.set_xlim(0, 1)
        ax.set_title('Depression Level Assessment', color='#e2e8f0', fontsize=12, pad=10)
        ax.tick_params(colors='#94a3b8', labelsize=9)
        ax.spines['bottom'].set_color('#475569')
        ax.spines['top'].set_color('#475569')
        ax.spines['right'].set_color('#475569')
        ax.spines['left'].set_color('#475569')

        # Add score indicator
        ax.text(0.5, current_score, '●', color='#e2e8f0', fontsize=16, ha='center', va='center')

        fig.tight_layout()

        # Embed in tkinter
        # canvas = FigureCanvasTkinter(fig, meter_frame)
        # canvas.get_tk_widget().pack(fill="both", expand=True, padx=15, pady=(0, 15))

    def setup_live_translation_tab(self):
        tab = self.tabview.tab("🎤 Live Translation")

        # Live translation interface
        live_frame = ctk.CTkFrame(
            tab,
            fg_color="#334155",
            corner_radius=12
        )
        live_frame.pack(fill="both", expand=True, padx=20, pady=20)

        title = ctk.CTkLabel(
            live_frame,
            text="Live Sign Language Translation",
            font=ctk.CTkFont(family="Montserrat", size=20, weight="bold"),
            text_color="#e2e8f0"
        )
        title.pack(pady=(20, 15))

        # Camera feed placeholder
        self.camera_frame = ctk.CTkFrame(
            live_frame,
            fg_color="#1e293b",
            corner_radius=8,
            height=400
        )
        self.camera_frame.pack(fill="x", padx=20, pady=10)
        self.camera_frame.pack_propagate(False)

        camera_placeholder = ctk.CTkLabel(
            self.camera_frame,
            text="📹\nCamera Feed\nClick 'Start Camera' to begin",
            font=ctk.CTkFont(size=16),
            text_color="#94a3b8"
        )
        camera_placeholder.pack(expand=True)

        # Control buttons
        controls_frame = ctk.CTkFrame(live_frame, fg_color="transparent")
        controls_frame.pack(pady=15)

        self.start_camera_btn = ctk.CTkButton(
            controls_frame,
            text="📹 Start Camera",
            command=self.start_camera,
            fg_color="#10b981",
            hover_color="#059669",
            font=ctk.CTkFont(weight="bold")
        )
        self.start_camera_btn.pack(side="left", padx=10)

        self.stop_camera_btn = ctk.CTkButton(
            controls_frame,
            text="⏹️ Stop Camera",
            command=self.stop_camera,
            fg_color="#ef4444",
            hover_color="#dc2626",
            font=ctk.CTkFont(weight="bold"),
            state="disabled"
        )
        self.stop_camera_btn.pack(side="left", padx=10)

        # Live translation output
        self.live_translation_text = ctk.CTkTextbox(
            live_frame,
            height=100,
            fg_color="#1e293b",
            text_color="#e2e8f0",
            font=ctk.CTkFont(size=14)
        )
        self.live_translation_text.pack(fill="x", padx=20, pady=(10, 20))
        self.live_translation_text.insert("0.0", "Live translation will appear here...")

    def create_features_section(self):
        # Features section
        features_frame = ctk.CTkFrame(
            self.main_frame,
            fg_color="#1e293b",
            corner_radius=12
        )
        features_frame.pack(fill="x", pady=(0, 20))

        features_title = ctk.CTkLabel(
            features_frame,
            text="Key Features",
            font=ctk.CTkFont(family="Montserrat", size=24, weight="bold"),
            text_color="#e2e8f0"
        )
        features_title.pack(pady=(20, 15))

        # Features grid
        features_grid = ctk.CTkFrame(features_frame, fg_color="transparent")
        features_grid.pack(pady=(0, 20))

        features_data = [
            ("🧠", "Advanced AI Analysis", "State-of-the-art machine learning models for accurate emotion detection and sign language recognition."),
            ("❤️", "Mental Health Support", "Comprehensive depression analysis and emotional well-being monitoring for better healthcare outcomes."),
            ("⚡", "Real-time Processing", "Instant analysis and feedback for seamless communication and immediate support when needed.")
        ]

        for i, (icon, title, description) in enumerate(features_data):
            feature_card = ctk.CTkFrame(
                features_grid,
                fg_color="#334155",
                corner_radius=8,
                width=400,
                height=150
            )
            feature_card.grid(row=0, column=i, padx=15, pady=10)
            feature_card.pack_propagate(False)

            icon_label = ctk.CTkLabel(
                feature_card,
                text=icon,
                font=ctk.CTkFont(size=24),
                text_color="#0ea5e9"
            )
            icon_label.pack(pady=(15, 5))

            title_label = ctk.CTkLabel(
                feature_card,
                text=title,
                font=ctk.CTkFont(family="Montserrat", size=14, weight="bold"),
                text_color="#e2e8f0"
            )
            title_label.pack()

            desc_label = ctk.CTkLabel(
                feature_card,
                text=description,
                font=ctk.CTkFont(size=11),
                text_color="#94a3b8",
                wraplength=350
            )
            desc_label.pack(pady=(5, 15), padx=15)

    def upload_video(self):
        file_path = filedialog.askopenfilename(
            title="Select Video File",
            filetypes=[("Video files", "*.mp4 *.avi *.mov *.mkv"), ("All files", "*.*")]
        )
        if file_path:
            self.video_path = file_path
            self.analyze_btn.configure(state="normal")
            messagebox.showinfo("Success", f"Video loaded: {file_path.split('/')[-1]}")

    def analyze_video(self):
        # Simulate video analysis
        self.progress_bar.set(0)
        self.analysis_data["is_processing"] = True

        def analysis_thread():
            for i in range(101):
                time.sleep(0.02)
                self.progress_bar.set(i/100)
                self.root.update_idletasks()

            # Simulate results
            self.analysis_data.update({
                "emotion": "Happy",
                "confidence": 87,
                "translation": "Hello, how are you today? I am feeling good and excited about this new technology.",
                "depression_score": 1,
                "is_processing": False
            })

            self.update_results()

        threading.Thread(target=analysis_thread, daemon=True).start()

    def update_results(self):
        # Update translation
        self.translation_text.delete("0.0", "end")
        self.translation_text.insert("0.0", self.analysis_data["translation"])

        # Update emotion
        self.emotion_label.configure(text=self.analysis_data["emotion"])
        self.confidence_label.configure(text=f"Confidence: {self.analysis_data['confidence']}%")

    def start_camera(self):
        self.is_recording = True
        self.start_camera_btn.configure(state="disabled")
        self.stop_camera_btn.configure(state="normal")

        # Clear camera frame and add live feed placeholder
        for widget in self.camera_frame.winfo_children():
            widget.destroy()

        live_label = ctk.CTkLabel(
            self.camera_frame,
            text="🔴 LIVE\nCamera is active\nAnalyzing sign language...",
            font=ctk.CTkFont(size=16),
            text_color="#10b981"
        )
        live_label.pack(expand=True)

        # Simulate live translation updates
        def live_translation_thread():
            translations = [
                "Detecting hand movements...",
                "Recognized: Hello",
                "Recognized: How are you?",
                "Recognized: I am fine, thank you",
                "Recognized: Nice to meet you"
            ]

            for translation in translations:
                if not self.is_recording:
                    break
                time.sleep(2)
                self.live_translation_text.delete("0.0", "end")
                self.live_translation_text.insert("0.0", translation)
                self.root.update_idletasks()

        threading.Thread(target=live_translation_thread, daemon=True).start()

    def stop_camera(self):
        self.is_recording = False
        self.start_camera_btn.configure(state="normal")
        self.stop_camera_btn.configure(state="disabled")

        # Reset camera frame
        for widget in self.camera_frame.winfo_children():
            widget.destroy()

        camera_placeholder = ctk.CTkLabel(
            self.camera_frame,
            text="📹\nCamera Feed\nClick 'Start Camera' to begin",
            font=ctk.CTkFont(size=16),
            text_color="#94a3b8"
        )
        camera_placeholder.pack(expand=True)

        self.live_translation_text.delete("0.0", "end")
        self.live_translation_text.insert("0.0", "Live translation will appear here...")

    def run(self):
        self.root.mainloop()

if __name__ == "__main__":
    app = AIAnalysisDashboard()
    app.run()
